In [1]:
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
from scipy.signal import find_peaks
from itertools import product
from tqdm import tqdm
import time
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import talib as ta
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
df=pd.read_csv('../../all_data_EUR_USD.csv')

In [3]:
df.set_index('time',inplace=True)

In [4]:
def calc_slope_high(high_vals):

    high_vals=[(val-min(high_vals))/(max(high_vals)-min(high_vals)) if (max(high_vals)-min(high_vals)) >0 else 0  for val in high_vals]
    
    grad_arr=[]
    
   

    

        

    
    for i in range(len(high_vals)):

        if i>0:
    
            if i==1:
    
                
                val=(high_vals[i]-high_vals[i-1])/1
                
                grad_arr.append(val)
                
    
            if (i>1):
    
                val=(high_vals[i]-high_vals[i-2])/2
               
                grad_arr.append(val)
                
    
                if i==(len(high_vals)-1):
    
                    
                    val2=(high_vals[i]-high_vals[i-1])/1
                   
                    grad_arr.append(val2)
                    
        
        
    return sum(grad_arr)
    

    

In [5]:
def calc_slope_low(low_vals):

    low_vals=[(val-min(low_vals))/(max(low_vals)-min(low_vals)) if (max(low_vals)-min(low_vals))>0 else 0 for val in low_vals  ]
    
    grad_arr=[]
    
   

  

        

    
    for i in range(len(low_vals)-1):

        if i>0:
    
            if i==1:
    
                
                val=(low_vals[i]-low_vals[i-1])/1
                
                grad_arr.append(val)
                
    
            if (i>1):
    
                val=(low_vals[i]-low_vals[i-2])/2
               
                grad_arr.append(val)
                
    
                if i==(len(low_vals)-1):
    
                    
                    val2=(low_vals[i]-low_vals[i-1])/1
                   
                    grad_arr.append(val2)
    
        
        
    return sum(grad_arr)
    

In [6]:
def calc_slope_rsi_low(rsi_low_vals):

    rsi_low_vals=[(val-min(rsi_low_vals))/(max(rsi_low_vals)-min(rsi_low_vals)) if (max(rsi_low_vals)-min(rsi_low_vals))>0 else 0  for val in rsi_low_vals]

    grad_arr=[]

    for i in range(len(rsi_low_vals)-1):
    
            if i>0:
        
                if i==1:
        
                    
                    val=(rsi_low_vals[i]-rsi_low_vals[i-1])/1
                    
                    grad_arr.append(val)
                    
        
                if (i>1):
        
                    val=(rsi_low_vals[i]-rsi_low_vals[i-2])/2
                   
                    grad_arr.append(val)
                    
        
                    if i==(len(rsi_low_vals)-1):
        
                        
                        val2=(rsi_low_vals[i]-rsi_low_vals[i-1])/1
                       
                        grad_arr.append(val2)
    
        
        
    return sum(grad_arr)

    

In [7]:
def calc_slope_rsi_high(rsi_high_vals):

    rsi_high_vals=[(val-min(rsi_high_vals))/(max(rsi_high_vals)-min(rsi_high_vals)) if (max(rsi_high_vals)-min(rsi_high_vals))>0 else 0  for val in rsi_high_vals]
    
    

    grad_arr=[]

    for i in range(len(rsi_high_vals)-1):
    
            if i>0:
        
                if i==1:
        
                    
                    val=(rsi_high_vals[i]-rsi_high_vals[i-1])/1
                    
                    grad_arr.append(val)
                    
        
                if (i>1):
        
                    val=(rsi_high_vals[i]-rsi_high_vals[i-2])/2
                   
                    grad_arr.append(val)
                    
        
                    if i==(len(rsi_high_vals)-1):
        
                        
                        val2=(rsi_high_vals[i]-rsi_high_vals[i-1])/1
                       
                        grad_arr.append(val2)
    
        
        
    return sum(grad_arr)


In [8]:
def calc_low_points(low_vals):

    temp_min=low_vals[0]
    

    for i in range(len(low_vals)-1):

         if low_vals[i]<=temp_min:

             temp_min=low_vals[i]


        

    if temp_min==low_vals[-2]:

       return 1

    else:

        return 0

            

In [9]:
def calc_high_points(high_vals, rsi_down_level=None):

    temp_max=high_vals[0]
    

    for i in range(len(high_vals)-1):

         if high_vals[i]>=temp_max:

             temp_max=high_vals[i]


        

    if temp_max==high_vals[-2]:

       return -1

    else:

        return 0

In [27]:
class RSI_divergence():

    def __init__(self, data):

        self.data=data

        self.params_range={'RSI_low_window':[10,14,20,28],
                           'RSI_high_window':[10,14,20,28],
                          'calc_points_window':[5,10,14,28,33]}

        self.possible_strats={'strategy_desc':'Strategy RSI divergence',
                              'div_point':{'name':'detect divergence RSI points',
                                          'positions':{'buy':'bull point',
                                                       'sell':'bear point'},
                                            'pos_columns':{}}}

    def create_params_combs(self):

        return list(product(*self.params_range.values()))

    def calc_indicator(self, RSI_low_window,RSI_high_window,calc_points_window):

        self.rsi_low_window=RSI_low_window
        self.rsi_high_window=RSI_high_window
        self.calc_points_window=calc_points_window
        

        df=self.data.copy()
        df['RSI_high']=ta.RSI(df['h'],RSI_high_window)
        df['RSI_low']=ta.RSI(df['l'],RSI_low_window)
        df['hh']=df['h'].rolling(calc_points_window).apply(calc_slope_high, raw=True, engine='numba')
        df['ll']=df['l'].rolling(calc_points_window).apply(calc_slope_low, raw=True, engine='numba')
        df['rsi_ll']=df['RSI_low'].rolling(calc_points_window).apply(calc_slope_rsi_low, raw=True, engine='numba')
        df['rsi_hh']=df['RSI_high'].rolling(calc_points_window).apply(calc_slope_rsi_high, raw=True, engine='numba')
        df['point_bear_div']=df['h'].rolling(RSI_high_window).apply(calc_high_points, raw=True, engine='numba')
        df['point_bull_div']=df['l'].rolling(RSI_low_window).apply(calc_low_points, raw=True, engine='numba')
        df['pot_bull_div']=df['ll']/df['rsi_ll']
        df['pot_bear_div']=df['hh']/df['rsi_hh']


        self.data=df.copy()

    def calc_position(self):

        self.pos_ch_colname=f'pos_ch_div_point_rlw_{self.rsi_low_window}_rhw_{self.rsi_high_window}_cpw_{self.calc_points_window}'
        self.pos_colname=f'pos_div_point_rlw_{self.rsi_low_window}_rhw_{self.rsi_high_window}_cpw_{self.calc_points_window}'
        

        df=self.data.copy()
        df['pos_ch_bull']=np.nan
        df['pos_ch_bull']=np.where((df['pot_bull_div']<0)&(df['pot_bull_div'].shift()>0),1,df['pos_ch_bull'])
        df['pos_ch_bull']=np.where((df['pot_bull_div']>0)&(df['pot_bull_div'].shift()<0),-2,df['pos_ch_bull'])
        df['pos_ch_bear']=np.nan
        df['pos_ch_bear']=np.where((df['pot_bear_div']<0)&(df['pot_bear_div'].shift()>0),-1,df['pos_ch_bear'])
        df['pos_ch_bear']=np.where((df['pot_bear_div']>0)&(df['pot_bear_div'].shift()<0),-2,df['pos_ch_bear'])
        df['pos_bull']=df['pos_ch_bull'].ffill()
        df['pos_bear']=df['pos_ch_bear'].ffill()
        df['pos_bull']=df['pos_bull'].map({-2:0,1:1})
        df['pos_bear']=df['pos_bear'].map({-2:0,-1:-1})
        df['pos_ch_bull_final']=np.nan
        df['pos_ch_bull_final']=np.where((df['point_bull_div']==1)&(df['pos_bull']==1),1,df['pos_ch_bull_final'])
        df['pos_ch_bear_final']=np.nan
        df['pos_ch_bear_final']=np.where((df['point_bear_div']==-1)&(df['pos_bear']==-1),-1,df['pos_ch_bear_final'])
        df['pos_ch_bull_final']=df['pos_ch_bull_final'].fillna(0)
        df['pos_ch_bear_final']=df['pos_ch_bear_final'].fillna(0)
        df[self.pos_ch_colname]=df['pos_ch_bull_final']+df['pos_ch_bear_final']
        df[self.pos_colname]=df[self.pos_ch_colname].replace(to_replace=0, method='ffill')
        

        self.data=df[[col for col in df.columns if col not in ['pos_ch_bull','pos_ch_bear','pos_bull','pos_bear','pos_ch_bull_final','pos_ch_bear_final',
                                                              'point_bear_div','point_bull_div','pot_bull_div','pot_bear_div']]]
        

        
                            

In [37]:
rsid=RSI_divergence(df)

In [38]:
comb=rsid.create_params_combs()

In [39]:
for c in tqdm(comb):

    rsid.calc_indicator(*c)
    rsid.calc_position()

100%|██████████| 80/80 [07:06<00:00,  5.33s/it]


In [40]:
rsid.data.columns

Index(['o', 'h', 'l', 'c', 'volume', 'complete', 'RSI_high', 'RSI_low', 'hh',
       'll',
       ...
       'pos_ch_div_point_rlw_28_rhw_28_cpw_5',
       'pos_div_point_rlw_28_rhw_28_cpw_5',
       'pos_ch_div_point_rlw_28_rhw_28_cpw_10',
       'pos_div_point_rlw_28_rhw_28_cpw_10',
       'pos_ch_div_point_rlw_28_rhw_28_cpw_14',
       'pos_div_point_rlw_28_rhw_28_cpw_14',
       'pos_ch_div_point_rlw_28_rhw_28_cpw_28',
       'pos_div_point_rlw_28_rhw_28_cpw_28',
       'pos_ch_div_point_rlw_28_rhw_28_cpw_33',
       'pos_div_point_rlw_28_rhw_28_cpw_33'],
      dtype='object', length=172)

In [41]:
rsid.data.to_csv('RSI_divergence_data.csv')

In [43]:
with open('rsi_divergence.json', "w") as f:
    json.dump(rsid.possible_strats, f)